# 01 · Carga y limpieza de datos

**Proyecto semestral IMT2200, Grupo 5:** *¿Mide el NEM lo mismo para todos?*

Carga las 16 tablas del MINEDUC (4 tipos × procesos 2023-2026), aplica las
decisiones de limpieza y guarda el panel unificado en
`data/processed/panel_completo.parquet`, que es la entrada de `02_eda.ipynb`. Las
funciones están en `src/` y los parámetros en `src/config.py`.

| Decisión | Contenido |
|---|---|
| 1 | Unicidad de `MRUN` y columnas comparables entre años |
| 2 | Postulaciones a una fila por estudiante |
| 3 | Panel por año (IP como base, `merge` `left`) |
| 4 | Tipos de datos y no respuesta de ingreso |
| 5 | Egreso reciente (`ANYO_DE_EGRESO = ANIO - 1`): se aplica en `02_eda` |
| 6 | Concatenación de los 4 años |

In [1]:
# Configuración: rutas, librerías y módulos del proyecto (carpeta src/).
import sys
from pathlib import Path

import pandas as pd

# Raíz del proyecto (funciona si el notebook se abre desde notebooks/ o
# desde la raíz) para poder importar los módulos de src/.
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))

# Los módulos de src/ se importan después de agregar la raíz al path; por
# eso llevan "noqa: E402" (PEP8 pide los imports al inicio del archivo).
from src.carga import cargar_tablas, comparar_columnas  # noqa: E402
from src.config import (ANIOS, ARCHIVO_PANEL,  # noqa: E402
                        COLUMNAS_PANEL, TABLAS)
from src.limpieza import (a_formato_largo, armar_panel,  # noqa: E402
                          limpiar_panel, resumir_postulacion)

## Carga de los datos

Los 16 CSV se leen **directo desde Google Drive** (URLs en `src/config.py`), sin
guardar copias locales, así que esta celda tarda algunos minutos.

In [2]:
# Lee los 16 CSV desde Google Drive y muestra cuántas filas tiene cada uno.
tablas = cargar_tablas(ANIOS)
ip, sd, ps, m = (tablas[sigla] for sigla in TABLAS)

filas = {sigla: {a: len(tablas[sigla][a]) for a in ANIOS} for sigla in TABLAS}
print('Filas por tabla (fila) y año (columna):')
display(pd.DataFrame(filas).T)

Leyendo IP-2026 desde Google Drive ...


Leyendo IP-2025 desde Google Drive ...


Leyendo IP-2024 desde Google Drive ...


Leyendo IP-2023 desde Google Drive ...


Leyendo SD-2026 desde Google Drive ...


Leyendo SD-2025 desde Google Drive ...


... (8 outputs left out)

Leyendo M-2024 desde Google Drive ...


Leyendo M-2023 desde Google Drive ...


Filas por tabla (fila) y año (columna):


,2026,2025,2024,2023
IP,320087,313759,309350,296812
SD,320087,313759,309350,296812
PS,1694081,1738995,1733891,187226
M,146873,140687,134492,128895


## Decisión 1: unicidad de `MRUN` y columnas entre años

- `IP`, `SD` y `M` tienen una fila por estudiante; `PS` tiene una por preferencia
  (hasta 20), por lo que se resume antes de unir (Decisión 2).
- Las diferencias de columnas entre años no afectan al análisis: **PS-2023** viene
  en formato ancho (se pasa a largo en la Decisión 2), IP-2023 y 2024 nombran
  distinto algunas columnas de invierno y del proceso anterior, M-2023 tiene otros
  nombres (de `M` solo se usa `MRUN`) y SD-2023 trae una encuesta adicional.

In [3]:
# Decisión 1: filas repetidas por MRUN (proceso 2026) y columnas que
# faltan o sobran en cada año respecto de 2026.
for sigla in TABLAS:
    repetidas = tablas[sigla][2026]['MRUN'].duplicated(keep=False).sum()
    print(f'{sigla}-2026: {repetidas} filas con MRUN repetido')
print()
for sigla in TABLAS:
    comparar_columnas(tablas[sigla], sigla)

IP-2026: 0 filas con MRUN repetido
SD-2026: 0 filas con MRUN repetido
PS-2026: 1685805 filas con MRUN repetido
M-2026: 0 filas con MRUN repetido

IP-2024: faltan 2 (ej.: ['MATE1_INV_ANTERIOR', 'MATE2_INV_ANTERIOR']), sobran 1 (ej.: ['MATE_INV_ANTERIOR'])
IP-2023: faltan 21 (ej.: ['CIEN_INV_ANTERIOR', 'CLEC_INV_ANTERIOR', 'CORRECTAS_INV_M1']), sobran 7 (ej.: ['CORRECTAS_INV_MATE', 'ERRADAS_INV_MATE', 'FORMA_INV_MATE'])
SD-2023: faltan 0 (ej.: []), sobran 29 (ej.: ['CONEXION_INSUFICIENTE', 'DISPOSITIVO_1', 'DISPOSITIVO_2'])
PS-2023: faltan 7 (ej.: ['COD_CARRERA_PREF', 'ESTADO_PREF', 'LUGAR_PREF']), sobran 82 (ej.: ['COD_CARRERA_PREF_01', 'COD_CARRERA_PREF_02', 'COD_CARRERA_PREF_03'])
M-2023: faltan 4 (ej.: ['LUGAR_EN_LA_LISTA', 'PREFERENCIA', 'PUNTAJE_PONDERADO']), sobran 3 (ej.: ['FECHA_MAT', 'NOMBRE_INSTITUCION', 'VIA_ADMISION'])


## Decisión 2: postulaciones a una fila por estudiante

`PS` se lleva a formato largo (en 2023 las preferencias vacías vienen con
`ESTADO_PREF = 0` y se eliminan) y se resume por estudiante. Códigos según el libro
de códigos de Postulación 2026 (`docs/`, Anexo I): `24` = en lista de seleccionados,
`25` = en lista de espera, `26` = seleccionado en una preferencia anterior.

- **Postulación válida:** alguna preferencia en 24, 25 o 26.
- **Seleccionado:** alguna preferencia en 24 o 26 (en PS-2023 a veces falta el 24).
- **Puntaje ponderado máximo:** el mayor `PTJE_PREF` entre sus preferencias.

**Control:** PS-2023 trae `SITUACION_POSTULANTE` (`P` = al menos una postulación en
selección o espera). Debería coincidir con la postulación válida; PS 2024-2026 no
traen esta columna.

In [4]:
# Decisión 2: PS a formato largo y una fila por estudiante.
ps_resumido = {a: resumir_postulacion(a_formato_largo(ps[a])) for a in ANIOS}
for a in ANIOS:
    n_sel = ps_resumido[a]['seleccionado'].sum()
    print(f'PS {a}: {len(ps_resumido[a])} postulantes, {n_sel} seleccionados')

PS 2026: 189399 postulantes, 161907 seleccionados
PS 2025: 189982 postulantes, 158218 seleccionados
PS 2024: 188707 postulantes, 150492 seleccionados
PS 2023: 187226 postulantes, 142902 seleccionados


In [5]:
# Control (solo 2023): postulación válida vs SITUACION_POSTULANTE = 'P'.
control = ps[2023][['MRUN', 'SITUACION_POSTULANTE']].merge(
    ps_resumido[2023], on='MRUN', how='left')
es_p = control['SITUACION_POSTULANTE'].astype(str).str.strip() == 'P'
coinciden = (es_p == control['postulacion_valida']).mean()
print(f'Coinciden en el {coinciden:.2%} de los {len(control)} postulantes')

Coinciden en el 100.00% de los 187226 postulantes


## Decisiones 3 y 4: panel por año, tipos y no respuesta de ingreso

- **Panel:** `IP` es el universo de inscritos, así que los `merge` son `left` para no
  perder a quienes no postularon o no se matricularon (es justamente la pregunta de
  acceso).
- **Tipos:** varias columnas vienen como texto por celdas en blanco; sin corregirlo,
  filtros como `ANYO_DE_EGRESO == 2025` fallan en silencio. Lo que no es número pasa
  a `NaN`.
- **Ingreso:** el código 99 es "no responde"; pasa a `NaN` sin eliminar filas y en el
  EDA se mantiene como categoría propia.

In [6]:
# Decisiones 3 y 4: un panel por año (IP + SD + PS + M) con tipos limpios.
paneles = {}
perdidos = {}
for a in ANIOS:
    panel = armar_panel(ip[a], sd[a], ps_resumido[a], m[a], a)
    paneles[a], perdidos[a] = limpiar_panel(panel)

print('Celdas no numéricas (en blanco) que pasaron a NaN, por columna y año:')
display(pd.DataFrame(perdidos))
sin_ingreso = paneles[2026]['INGRESO_PERCAPITA_GRUPO_FA'].isna().mean()
print(f'2026: {sin_ingreso:.1%} no declara ingreso (código 99)')

Celdas no numéricas (en blanco) que pasaron a NaN, por columna y año:


,2026,2025,2024,2023
RBD,3200,5102,4932.0,2658.0
CODIGO_ENS,3200,5102,4932.0,2658.0
DEPENDENCIA,3200,5102,4932.0,2658.0
SEXO,0,0,0.0,0.0
ANYO_DE_EGRESO,218,137,189.0,334.0
PROMEDIO_NOTAS,0,0,0.0,0.0
PTJE_NEM,0,0,0.0,0.0
INGRESO_PERCAPITA_GRUPO_FA,0,0,0.0,0.0
CLEC_REG_ACTUAL,19496,19664,21753.0,21598.0
CLEC_INV_ACTUAL,288070,282692,279286.0,263432.0


2026: 27.6% no declara ingreso (código 99)


## Decisión 6: concatenación y guardado

Se apilan los cuatro años y se guardan solo las columnas que usa el EDA, en formato
Parquet (más liviano que CSV y conserva los tipos de datos).

In [7]:
# Decisión 6: apila los 4 años y guarda el panel para 02_eda.ipynb.
panel_completo = pd.concat(paneles.values(), ignore_index=True)
columnas = [c for c in COLUMNAS_PANEL if c in panel_completo.columns]
ARCHIVO_PANEL.parent.mkdir(parents=True, exist_ok=True)
panel_completo[columnas].to_parquet(ARCHIVO_PANEL, index=False)
print(f'Guardado {ARCHIVO_PANEL.name}: {len(panel_completo)} inscritos, '
      f'{len(columnas)} columnas')

Guardado panel_completo.parquet: 1240008 inscritos, 32 columnas
